# LP Quality Evaluation - 0227

分别对以下两个文件进行质量评估：
1. `UHRS_Task_lp_quality_labeling_0227-PromptRefiner_Internal100.tsv`
2. `UHRS_Task_lp_quality_labeling_0224.tsv` (参照基准)

---
## File 1: PromptRefiner_Internal100 (0227)

In [ ]:
import pandas as pd
import os

# ================= 配置区域 =================
input_file_1 = r"D:\Data\T2I\AutoGenValidation\UHRS_Task_lp_quality_labeling_0227-PromptRefiner_Internal100.tsv"

priority_dimensions = [
    'TextLogoClarity', 
    'RealisticPhysicallyPlausible', 
    'BrightnessContrastColorSaturation', 
    'Imageclarity', 
    'SubjectClarity', 
    'CompositionLayout'
]

# ================= 1. 数据清洗与聚合 =================
df1 = pd.read_csv(input_file_1, sep='\t', header=0)

df1['ImgUrl'] = df1['ImgUrl'].astype(str).str.strip()
df1['FinalDecision'] = df1['FinalDecision'].astype(str).str.strip()

print(f"File 1 数据总量: {len(df1)} 条")
print(f"列名: {df1.columns.tolist()}")
df1.head(3)

In [ ]:
# 数字化维度得分
for dim in priority_dimensions:
    df1[dim] = df1[dim].astype(str).str.strip()
    df1[f"{dim}_score"] = (df1[dim] == 'Bad').astype(int)

# 数字化总投票
df1['Final_score'] = (df1['FinalDecision'] == 'Bad').astype(int)

# ---【按 ImgUrl 聚合所有人的投票】---
agg_rules = {f"{dim}_score": 'sum' for dim in priority_dimensions}
agg_rules['Final_score'] = 'sum'

grouped1 = df1.groupby('ImgUrl').agg(agg_rules).reset_index()

# ================= 2. 锁定 Bad 图片池 =================
bad_pool1 = grouped1[grouped1['Final_score'] >= 2].copy()

print(f"--- File 1 最终统计 ---")
print(f"聚合后唯一 URL 总数: {len(grouped1)}")
print(f"最终判定为 Bad 的图片数: {len(bad_pool1)}")
print(f"Bad Rate: {len(bad_pool1)/len(grouped1)*100:.1f}%")

In [ ]:
# ================= 3. 各维度 Bad 分布 =================
print("=== File 1: 各维度 Bad 图片数 (Bad 票数 >= 2) ===")
processed_urls1 = set()

for dim in priority_dimensions:
    col_name = f"{dim}_score"
    current_batch = bad_pool1[
        (~bad_pool1['ImgUrl'].isin(processed_urls1)) & 
        (bad_pool1[col_name] >= 2)
    ].copy()
    print(f"维度 [{dim}]: {len(current_batch)} 张")
    for _, row in current_batch.iterrows():
        processed_urls1.add(row['ImgUrl'])

remaining_bad1 = bad_pool1[~bad_pool1['ImgUrl'].isin(processed_urls1)].copy()
if len(remaining_bad1) > 0:
    print(f"原因不一致: {len(remaining_bad1)} 张")

print(f"\nBad 图片 URL 列表:")
for url in bad_pool1['ImgUrl'].tolist():
    print(url)

In [ ]:
# ================= 4. 各维度 Bad Rate 汇总（不去重）=================
print("=== File 1: 各维度 Bad Rate ===")
total_urls1 = len(grouped1)
dim_stats1 = {}
for dim in priority_dimensions:
    col_name = f"{dim}_score"
    bad_count = (grouped1[col_name] >= 2).sum()
    dim_stats1[dim] = bad_count
    print(f"  {dim}: {bad_count} / {total_urls1} = {bad_count/total_urls1*100:.1f}%")

print(f"\n  Overall Bad: {len(bad_pool1)} / {total_urls1} = {len(bad_pool1)/total_urls1*100:.1f}%")

---
## File 2: lp_quality_labeling_0224 (参照基准)

In [ ]:
# ================= 配置区域 =================
input_file_2 = r"D:\Data\T2I\AutoGenValidation\AdsT2I_0131-2\UHRS_Task_lp_quality_labeling_0224.tsv"

# ================= 1. 数据清洗与聚合 =================
df2 = pd.read_csv(input_file_2, sep='\t', header=0)

df2['ImgUrl'] = df2['ImgUrl'].astype(str).str.strip()
df2['FinalDecision'] = df2['FinalDecision'].astype(str).str.strip()

print(f"File 2 数据总量: {len(df2)} 条")
print(f"列名: {df2.columns.tolist()}")
df2.head(3)

In [ ]:
# 数字化维度得分
for dim in priority_dimensions:
    df2[dim] = df2[dim].astype(str).str.strip()
    df2[f"{dim}_score"] = (df2[dim] == 'Bad').astype(int)

df2['Final_score'] = (df2['FinalDecision'] == 'Bad').astype(int)

agg_rules = {f"{dim}_score": 'sum' for dim in priority_dimensions}
agg_rules['Final_score'] = 'sum'

grouped2 = df2.groupby('ImgUrl').agg(agg_rules).reset_index()

# ================= 2. 锁定 Bad 图片池 =================
bad_pool2 = grouped2[grouped2['Final_score'] >= 2].copy()

print(f"--- File 2 最终统计 ---")
print(f"聚合后唯一 URL 总数: {len(grouped2)}")
print(f"最终判定为 Bad 的图片数: {len(bad_pool2)}")
print(f"Bad Rate: {len(bad_pool2)/len(grouped2)*100:.1f}%")

In [ ]:
# ================= 3. 各维度 Bad 分布 =================
print("=== File 2: 各维度 Bad 图片数 (Bad 票数 >= 2) ===")
processed_urls2 = set()

for dim in priority_dimensions:
    col_name = f"{dim}_score"
    current_batch = bad_pool2[
        (~bad_pool2['ImgUrl'].isin(processed_urls2)) & 
        (bad_pool2[col_name] >= 2)
    ].copy()
    print(f"维度 [{dim}]: {len(current_batch)} 张")
    for _, row in current_batch.iterrows():
        processed_urls2.add(row['ImgUrl'])

remaining_bad2 = bad_pool2[~bad_pool2['ImgUrl'].isin(processed_urls2)].copy()
if len(remaining_bad2) > 0:
    print(f"原因不一致: {len(remaining_bad2)} 张")

print(f"\nBad 图片 URL 列表:")
for url in bad_pool2['ImgUrl'].tolist():
    print(url)

In [ ]:
# ================= 4. 各维度 Bad Rate 汇总（不去重）=================
print("=== File 2: 各维度 Bad Rate ===")
total_urls2 = len(grouped2)
dim_stats2 = {}
for dim in priority_dimensions:
    col_name = f"{dim}_score"
    bad_count = (grouped2[col_name] >= 2).sum()
    dim_stats2[dim] = bad_count
    print(f"  {dim}: {bad_count} / {total_urls2} = {bad_count/total_urls2*100:.1f}%")

print(f"\n  Overall Bad: {len(bad_pool2)} / {total_urls2} = {len(bad_pool2)/total_urls2*100:.1f}%")

---
## 对比汇总

In [ ]:
# ================= 两个文件对比 =================
print("=== 对比汇总 ===")
print(f"{'维度':<40} {'File1(0227)':>15} {'File2(0224)':>15}")
print("-" * 72)

for dim in priority_dimensions:
    col = f"{dim}_score"
    c1 = (grouped1[col] >= 2).sum()
    c2 = (grouped2[col] >= 2).sum()
    r1 = f"{c1}/{len(grouped1)} ({c1/len(grouped1)*100:.1f}%)"
    r2 = f"{c2}/{len(grouped2)} ({c2/len(grouped2)*100:.1f}%)"
    print(f"{dim:<40} {r1:>15} {r2:>15}")

print("-" * 72)
overall1 = f"{len(bad_pool1)}/{len(grouped1)} ({len(bad_pool1)/len(grouped1)*100:.1f}%)"
overall2 = f"{len(bad_pool2)}/{len(grouped2)} ({len(bad_pool2)/len(grouped2)*100:.1f}%)"
print(f"{'Overall Bad':<40} {overall1:>15} {overall2:>15}")